In [0]:
#Config details
storage_account = "jayanstgact"
container = "landing-zone"
sp_secret_scope_name = "sp_secret_scope"
sp_client_secret_key = "sp_client_secret"
sp_client_secret = dbutils.secrets.get(scope = "sp_secret_scope", key = "sp_client_secret")
sp_client_id = "4805ff50-0e01-488a-ba14-d1dfc28959ef"
tenant_id = "d282161d-31c8-4698-be9e-26f18fb7660f"

# Spark Auth Config to read the files from ADLS
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    sp_client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    sp_client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)



In [0]:
from pyspark.sql.functions import current_timestamp, current_date
from datetime import datetime

# Bronze Table
bronze_table = "soundhar_ws.insurance_bronze.policyholder_raw"

# Storage Details
storage_account = "jayanstgact"
container = "landing-zone"

# Widget parameter
dbutils.widgets.text("filename", "dim_policyholder_20240101102227.parquet")
file_name = dbutils.widgets.get("filename")

# Today's date
today = datetime.now().strftime("%Y%m%d")

# Extract date from filename
file_date = file_name.split("_")[-1][:8]

# Optional validation
if file_date != today:
    raise Exception(
        f"File date ({file_date}) does not match today's date ({today})"
    )

# File Path
bronze_src_path = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    f"dim_policyholder/{file_name}"
)

# Read parquet file
df = spark.read.format("parquet").load(bronze_src_path)

# Bronze Layer - As Is
bronze_df = (
    df.withColumn("Ingestion_time", current_timestamp())
)

#display(bronze_df)

# Write to Bronze Delta Table
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print(f"Successfully loaded {file_name} into {bronze_table}")


In [0]:
# Incrementally loading the data from bronze to silver

from pyspark.sql.functions import col, sha2, upper, trim

silver_table = "soundhar_ws.insurance_silver.dim_policyholder_clean"

if spark.catalog.tableExists(silver_table):
    silver_df = spark.sql("""
SELECT *
FROM soundhar_ws.insurance_bronze.policyholder_raw
WHERE ingestion_time >
(
    SELECT COALESCE(
        MAX(ingestion_time),
        TIMESTAMP('1900-01-01')
    )
    FROM soundhar_ws.insurance_silver.dim_policyholder_clean
)
""")
else:
    silver_df = spark.read.table(bronze_table)
    

filter_bad_data = silver_df.filter(col("ph_id").isNotNull())

domain_standardization = filter_bad_data.withColumn("Tier", upper(trim(col("Tier"))))   

PII_Security = (domain_standardization.withColumn("Name_masked", sha2(col("name"), 256))
                .drop("name").select("PH_ID", "Name_masked", "Risk_Score", "Tier", "Status", "Load_dt","Ingestion_time")
)
                
#display(PII_Security)

(
    PII_Security.write
    .format("delta")
    .mode("append")
    .saveAsTable(silver_table)
)


In [0]:
# Gold Aggregation Layer - SCD Type 2

from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import TimestampType

gold_table = "soundhar_ws.insurance_gold.dim_policyholder_gold"

if spark.catalog.tableExists(gold_table):
    gold_df = spark.sql("""
MERGE INTO insurance_gold.dim_policyholder_gold t
USING insurance_silver.dim_policyholder_clean s
ON t.ph_id = s.ph_id
AND t.is_current = true

WHEN MATCHED AND (
       t.Name_masked <> s.Name_masked
    OR t.risk_score <> s.risk_score
    OR t.tier <> s.tier
    OR t.status <> s.status
)
THEN UPDATE SET
    t.is_current = false,
    t.end_date = current_timestamp();
""")   
else:
    gold_df = spark.read.table(silver_table)
    
gold_df = ( silver_df 
    .withColumn("effective_date", current_timestamp()) 
    .withColumn("end_date", lit(None).cast(TimestampType())) 
    .withColumn("is_current", lit(True))
)

#display(gold_df)

gold_df.write.mode("append").saveAsTable(gold_table)


